# 04 · Camada Silver — Dimensões conformadas

## 1. Objetivo e método

Dimensão conformada é a que várias tabelas de fato compartilham. Ela precisa existir antes delas e ter dono único, senão cada fonte constrói a sua e o modelo perde a capacidade de cruzar métricas — que é justamente o que este trabalho faz ao comparar seis dimensões de qualidade da mesma distribuidora.

Este notebook produz `dim_distribuidora`, que as seis métricas usam. `dim_conjunto` fica na Silver de continuidade, por ser específica daquela fonte: nenhuma outra métrica desce ao nível do conjunto de unidades consumidoras.

### Por que o porte vem da base de continuidade

O critério de grande porte exige uma contagem de unidades consumidoras, e cada fonte da ANEEL publica a sua, com universos que não coincidem. A escolha é a base de continuidade, pelo campo `NumCon`, por três motivos: cobre toda a série sem falha, tem grão mensal por conjunto, e é a mesma contagem que a norma usa para ponderar o indicador — ou seja, o universo que define o porte é o mesmo que pesa o resultado.

Essa escolha não contradiz a regra de que cada métrica usa o denominador da própria fonte. São coisas distintas: aqui o número classifica a empresa, lá ele divide o indicador.

### Ordem de execução

Este notebook roda depois da Bronze e antes de qualquer Silver de fonte.


## 2. Configuração

In [ ]:
import os
import sys

from pyspark.sql import functions as F

# Walk up from the working directory until the folder holding `src` is found,
# so the notebook works at any depth inside notebooks/
REPO_ROOT = os.getcwd()
while not os.path.isdir(os.path.join(REPO_ROOT, "src")):
    parent = os.path.dirname(REPO_ROOT)
    if parent == REPO_ROOT:
        raise FileNotFoundError("Repository root with a src folder not found above " + os.getcwd())
    REPO_ROOT = parent
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from src.config import CATALOG, SCHEMA_BRONZE, SCHEMA_SILVER

BRONZE = f"{CATALOG}.{SCHEMA_BRONZE}"
SILVER = f"{CATALOG}.{SCHEMA_SILVER}"

# Reference year for the size criterion: the first year of the series, so that the
# universe is fixed at the start of the period being compared.
ANO_REFERENCIA_PORTE = 2022
CORTE_UCS = 400_000

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA_SILVER}")

print(f"Origem..........: {BRONZE}")
print(f"Destino.........: {SILVER}")
print(f"Porte...........: NumCon de dezembro de {ANO_REFERENCIA_PORTE}, "
      f"corte em {CORTE_UCS:,} UCs")

## 3. `dim_distribuidora`

Grão CNPJ. Carrega o critério de porte como atributo, não como filtro — o recorte do universo é decisão analítica e acontece na Gold.

#### Ler o peso direto da Bronze:

Como este notebook antecede a Silver de continuidade, a normalização mínima acontece aqui: identificadores para texto de largura fixa e remoção das linhas duplicadas, sem as quais o peso de alguns conjuntos contaria duas vezes.

É a mesma normalização que a Silver de continuidade aplica à série inteira. Aqui ela alcança apenas o ano de referência e a parcela `NumCon`, que é o que o porte exige.

In [ ]:
CHAVE = ["num_cnpj", "ide_conjunto", "ano", "mes", "sig_indicador"]

bronze = spark.table(f"{BRONZE}.continuity_indicators")

normalizado = (bronze
    .withColumn("num_cnpj", F.lpad(F.col("NumCNPJ").cast("string"), 14, "0"))
    .withColumn("ide_conjunto", F.lpad(F.col("IdeConjUndConsumidoras").cast("string"), 5, "0"))
    .withColumn("ano", F.col("AnoIndice").cast("int"))
    .withColumn("mes", F.col("NumPeriodoIndice").cast("int"))
    .withColumn("sig_indicador", F.trim("SigIndicador"))
    .withColumn("sig_agente", F.trim("SigAgente"))
    .withColumn("valor", F.col("VlrIndiceEnviado").cast("double"))
    .select("num_cnpj", "sig_agente", "ide_conjunto", "ano", "mes",
            "sig_indicador", "valor")
    .dropDuplicates(CHAVE))

peso = (normalizado
    .filter((F.col("sig_indicador") == "NumCon") &
            (F.col("ano") == ANO_REFERENCIA_PORTE))
    .select("num_cnpj", "ide_conjunto", "mes", "valor"))

ucs_mes = (peso
    .groupBy("num_cnpj", "mes")
    .agg(F.sum("valor").alias("ucs_no_mes")))

print(f"distribuidoras com peso em {ANO_REFERENCIA_PORTE}: "
      f"{ucs_mes.select('num_cnpj').distinct().count()}")
print(f"meses encontrados: {sorted(r['mes'] for r in ucs_mes.select('mes').distinct().collect())}")

#### Compor a dimensão:

Dois critérios de porte convivem na tabela. Dezembro fixa o universo num instante e é o que prevalece; a média dos doze meses entra ao lado como sensibilidade, porque dezembro pode cair num mês de reestruturação de conjuntos. Quando os dois discordam sobre quem passa do corte, a empresa é marcada como fronteira e a decisão fica visível em vez de escondida.

In [ ]:
ucs_dezembro = (ucs_mes.filter(F.col("mes") == 12)
    .select("num_cnpj", F.col("ucs_no_mes").alias("qtd_ucs_dezembro")))

ucs_media = (ucs_mes
    .groupBy("num_cnpj")
    .agg(F.round(F.avg("ucs_no_mes"), 0).cast("long").alias("qtd_ucs_media")))

# The label check in notebook 03 found no CNPJ carrying more than one sigla, so max()
# picks the only value there is.
siglas = (normalizado.select("num_cnpj", "sig_agente").distinct()
    .groupBy("num_cnpj").agg(F.max("sig_agente").alias("sig_agente")))

dim_distribuidora = (siglas
    .join(ucs_dezembro, "num_cnpj", "left")
    .join(ucs_media, "num_cnpj", "left")
    .withColumn("grande_porte", F.coalesce(F.col("qtd_ucs_dezembro"), F.lit(0)) >= CORTE_UCS)
    .withColumn("grande_porte_media", F.coalesce(F.col("qtd_ucs_media"), F.lit(0)) >= CORTE_UCS)
    # Fronteira marks the companies the two criteria disagree about; December decides.
    .withColumn("fronteira", F.col("grande_porte") != F.col("grande_porte_media"))
    .withColumn("ano_referencia", F.lit(ANO_REFERENCIA_PORTE)))

(dim_distribuidora.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER}.dim_distribuidora"))

print(f"distribuidoras..............: {dim_distribuidora.count()}")
print(f"grande porte por dezembro...: {dim_distribuidora.filter('grande_porte').count()}")
print(f"grande porte pela media.....: {dim_distribuidora.filter('grande_porte_media').count()}")
print(f"na fronteira................: {dim_distribuidora.filter('fronteira').count()}")

display(dim_distribuidora
        .filter(F.col("grande_porte") | F.col("fronteira"))
        .select("sig_agente", "num_cnpj", "qtd_ucs_dezembro", "qtd_ucs_media",
                "grande_porte", "grande_porte_media", "fronteira")
        .orderBy(F.col("qtd_ucs_dezembro").desc()))

In [ ]:
COMENTARIOS_DX = {
    "num_cnpj": "CNPJ da distribuidora, texto de 14 caracteres com zeros a esquerda",
    "sig_agente": "Sigla da distribuidora conforme publicada pela ANEEL",
    "qtd_ucs_dezembro": "Soma de NumCon dos conjuntos em dezembro do ano de referencia",
    "qtd_ucs_media": "Media mensal da soma de NumCon no ano de referencia",
    "grande_porte": "Verdadeiro quando qtd_ucs_dezembro atinge o corte de 400 mil UCs",
    "grande_porte_media": "Mesmo corte aplicado a qtd_ucs_media, como sensibilidade",
    "fronteira": "Verdadeiro quando os dois criterios discordam sobre o porte",
    "ano_referencia": "Ano civil usado para medir o porte",
}

spark.sql(f"""COMMENT ON TABLE {SILVER}.dim_distribuidora IS
    'Dimensao conformada das distribuidoras, compartilhada por todas as metricas do
     trabalho. Carrega o criterio de porte como atributo; o recorte do universo e
     aplicado na Gold pela flag grande_porte, nao por filtro nesta tabela.'""")

for coluna, texto in COMENTARIOS_DX.items():
    spark.sql(f"ALTER TABLE {SILVER}.dim_distribuidora "
              f"ALTER COLUMN {coluna} COMMENT '{texto}'")

print(f"comentarios aplicados: {len(COMENTARIOS_DX)} colunas")

## 4. Validação

Três verificações. Uma dimensão conformada com chave duplicada corromperia toda tabela de fato que a ela se junte, então o teste de unicidade é o mais importante desta etapa.

In [ ]:
dim_lida = spark.table(f"{SILVER}.dim_distribuidora")

testes = []

linhas = dim_lida.count()
unicas = dim_lida.select("num_cnpj").distinct().count()
testes.append(("chave unica por CNPJ",
               linhas == unicas,
               f"{linhas} linhas para {unicas} CNPJs distintos"))

largura = dim_lida.filter(F.length("num_cnpj") != 14).count()
testes.append(("CNPJ com largura fixa",
               largura == 0,
               f"{largura} linhas fora de 14 caracteres"))

sem_porte = dim_lida.filter(F.col("qtd_ucs_dezembro").isNull()).count()
testes.append(("porte apurado para toda distribuidora",
               sem_porte == 0,
               f"{sem_porte} distribuidoras sem peso em dezembro de "
               f"{ANO_REFERENCIA_PORTE}"))

for nome, passou, detalhe in testes:
    print(f"[{'OK' if passou else 'FALHOU':<7}] {nome:<45} {detalhe}")

if all(p for _, p, _ in testes):
    print("\nDimensoes conformadas validadas.")
else:
    print("\nHa teste sem passar; corrigir antes de seguir para as Silvers de fonte.")

## Pendências documentadas

| Item | Situação | O que falta |
|---|---|---|
| Distribuidora sem peso no ano de referência | Verificada na validação | Definir o tratamento caso surja uma empresa que entrou na série depois de 2022 |
| Dimensão de tempo | Não criada | Avaliar se o modelo ganha com uma `dim_tempo` explícita ou se ano e mês como atributos do fato bastam para a análise |


## Autoavaliação desta etapa

A preencher após a execução.